# Airflow Project: Production Data Pipeline System


## Prerequisites

- Complete **`04_pipeline_integration.ipynb`** first.
- Start Airflow from **[`airflow/README.md`](../../airflow/README.md)**.
- Ensure Postgres table exists via **`../../projects/batch_pipeline/schema.sql`**.


## Problem Statement

You are a Data Engineer building a production pipeline.

Requirements:
- Fetch data from APIs
- Clean and transform data
- Load into a database
- Run analytics
- Automate with Airflow

Goal:
👉 Build a fully automated system.


## System Architecture

**Users API → Transform → Load → Analytics**

Airflow controls:
- Scheduling
- Execution
- Monitoring


## DAG Design

Create file: **`../../airflow/dags/production_pipeline_dag.py`**.

```python
from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime, timedelta
import sys

sys.path.append("/opt/airflow/projects/batch_pipeline")

from etl_pipeline import extract, transform, load

default_args = {
    "retries": 2,
    "retry_delay": timedelta(seconds=10)
}

with DAG(
    dag_id="production_pipeline",
    start_date=datetime(2024, 1, 1),
    schedule_interval="@daily",
    catchup=False,
    default_args=default_args
) as dag:

    def extract_task(**context):
        data = extract()
        context['ti'].xcom_push(key='raw_data', value=data)

    def transform_task(**context):
        raw = context['ti'].xcom_pull(key='raw_data', task_ids='extract')
        cleaned = transform(raw)
        context['ti'].xcom_push(key='cleaned_data', value=cleaned)

    def load_task(**context):
        data = context['ti'].xcom_pull(key='cleaned_data', task_ids='transform')
        load(data)

    def analytics_task():
        print("Running analytics...")

    extract_op = PythonOperator(
        task_id="extract",
        python_callable=extract_task
    )

    transform_op = PythonOperator(
        task_id="transform",
        python_callable=transform_task
    )

    load_op = PythonOperator(
        task_id="load",
        python_callable=load_task
    )

    analytics_op = PythonOperator(
        task_id="analytics",
        python_callable=analytics_task
    )

    extract_op >> transform_op >> load_op >> analytics_op
```

The repository DAG includes retries, explicit XCom hand-offs, logging, and a validation step before load.


## Features

- DAG-based orchestration
- Multi-step pipeline
- Retry mechanism
- XCom data passing
- Logging
- Database integration


## Practice

1. Add a validation task.
2. Add a parallel task (logging / monitoring).
3. Trigger the DAG and observe the flow in Graph/Grid views.


## Assignment

Upgrade the pipeline:
- Add incremental processing
- Add failure simulation
- Add an alert mechanism

Bonus:
- Add a second DAG (analytics DAG)


## How to Explain This Project

"I built a production-style data pipeline using Airflow.

- Extracted data from APIs
- Transformed and cleaned it
- Loaded into PostgreSQL
- Automated using DAGs
- Added retries, logging, and monitoring

This pipeline runs daily and handles failures gracefully."


## Interview Questions

1. How do you design Airflow pipelines?
2. What is XCom?
3. How do you handle failures?
4. How do you scale Airflow pipelines?


## What you just built

This is your strongest project so far:

- Multi-step DAG
- Real ETL integration
- Automation plus scheduling
- Production concepts

👉 This is very close to real company workflow automation.


---

**Phase 5 complete.**

Next: **Phase 6 — Kafka (real-time systems)**
- Streaming pipelines
- Event-driven architecture
- Real-time processing


**Reality check:** you now build pipelines, automate them, and run production-style workflows with failure handling.


## Your tasks

- [ ] Enable **`production_pipeline`** DAG in Airflow and trigger it end-to-end.
- [ ] Verify task order: `extract -> transform -> validate -> load -> analytics`.
- [ ] Confirm retries / logs in task instances.
- [ ] Validate DB rows in `users` and write a 4-line interview story.
